# Imports de librerias

In [0]:
from pyspark.sql.functions import col, to_timestamp, lower, trim, datediff, expr, length
from pyspark.sql.types import IntegerType

# Lectura de la Tabla Bronce olist_order_reviews

In [0]:
# creo el df con la tabla bronce de olist_customers
df = spark.table("`catalog_brazilian-e-commerce`.bronze.olist_order_reviews_dataset")

In [0]:
df.display()

#Transformaciones

In [0]:

# Se modifica el tipo de dato de las columnas
df_clean = (
    df
    .withColumn("review_creation_date", expr("try_cast(review_creation_date as timestamp)"))
    .withColumn("review_answer_timestamp", expr("try_cast(review_answer_timestamp as timestamp)"))
    .withColumn("review_score", expr("try_cast(review_score as int)"))
)


In [0]:
# limpieza de duplicados y nulos
df_clean = (
    df_clean
    .dropDuplicates(["review_id"])
    #.dropna(subset=["review_id", "order_id", "review_score"])
)

In [0]:
# normalizacion de texto a minuscula
df_clean = (
    df_clean
    .withColumn("review_comment_title", trim(lower(col("review_comment_title"))))
    .withColumn("review_comment_message", trim(lower(col("review_comment_message"))))
)


In [0]:

# Regla principal: score entre 1 y 5
# se separan 2 df con la data valida e invaldia tomando la "clean data"


df_valid = df_clean.filter(
    (col("review_score").between(1, 5)) &
    (col("order_id").isNotNull()) &
    (length(col("order_id")) > 10) &
    (col("review_creation_date").isNotNull()) &
    (col("review_id").isNotNull()) &
    (length(col("review_id")) > 10)
)




df_invalid = df_clean.filter(
    (col("review_score").isNull()) |
    (~col("review_score").between(1, 5)) |
    (col("order_id").isNull()) |
    (length(col("order_id")) <= 10) |
    (col("review_creation_date").isNull()) |
    (col("review_id").isNull()) |
    (trim(col("review_id")) == "") |
    (length(col("review_id")) <= 10)
)



In [0]:
# columna nueva con el tiempo de respuesta a los comentarios
df_valid = df_valid.withColumn(
    "response_time_days",
    datediff(col("review_answer_timestamp"), col("review_creation_date"))
)


In [0]:
total_original = df.count()
total_count = df_clean.count()
valid_count = df_valid.count()
invalid_count = df_invalid.count()

print(f"Total registros df original: {total_original}")
print(f"Total registros: {total_count}")
print(f"Validos: {valid_count}")
print(f"Invalidos: {invalid_count}")

# Crear la tabla Silver de olist_order_reviews

In [0]:
df_valid.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.olist_order_reviews")

In [0]:
# hago la misma delta table pero para los datos malos
df_invalid.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.olist_order_reviews_bad")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.olist_order_reviews

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.olist_order_reviews_bad